# Stage B continual-pretraining baseline gate

This notebook is the required **zero-update evaluation gate** before medical continual pretraining. It restores the promoted Stage A checkpoint, verifies the disjoint Stage B data, and records medical and general validation baselines. It does not train the model.

## Before running

1. In Colab, choose **Runtime → Change runtime type → GPU**.
2. Upload `stage-b-data.tar` to `MyDrive/medical-slm/stage-b-data.tar`.
3. Keep the promoted checkpoint at `MyDrive/medical-slm-runs/stage_a/checkpoints/checkpoint_00007250`.
4. Push the current repository changes to GitHub so the clone includes Stage B support.

T4 will use FP16 with a gradient scaler; A100 will use BF16. Baseline evaluation itself performs no backward pass.

In [ ]:
from google.colab import drive
from pathlib import Path
import os
import shutil
import subprocess

drive.mount('/content/drive')

REPOSITORY_URL = 'https://github.com/moshiur00/small-language-model-for-medical-domain.git'
REPOSITORY = Path('/content/medical-slm')
DATA_ARCHIVE = Path('/content/drive/MyDrive/medical-slm/stage-b-data.tar')
DRIVE_PARENT = Path('/content/drive/MyDrive/medical-slm-runs/stage_a/checkpoints/checkpoint_00007250')
LOCAL_PARENT = Path('/content/stage_a_parent/checkpoint_00007250')
LOCAL_REPORT = Path('/content/stage_b/stage_b_baseline.json')
DRIVE_REPORT = Path('/content/drive/MyDrive/medical-slm-runs/stage_b/stage_b_baseline.json')

assert DATA_ARCHIVE.is_file(), f'Missing {DATA_ARCHIVE}'
assert (DRIVE_PARENT / 'manifest.json').is_file(), f'Missing promoted Stage A checkpoint: {DRIVE_PARENT}'
print('Inputs found.')

## Download code and install the project

In [ ]:
if not (REPOSITORY / '.git').is_dir():
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(REPOSITORY)], check=True)
else:
    subprocess.run(['git', '-C', str(REPOSITORY), 'pull', '--ff-only'], check=True)

subprocess.run(['python', '-m', 'pip', 'install', '-q', '-e', str(REPOSITORY) + '[dev]'], check=True)
os.chdir(REPOSITORY)
print('Repository revision:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())

## Restore the data and promoted Stage A parent

In [ ]:
subprocess.run(['tar', '-xf', str(DATA_ARCHIVE), '-C', str(REPOSITORY)], check=True)
LOCAL_PARENT.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(DRIVE_PARENT, LOCAL_PARENT, dirs_exist_ok=True)

stage_b_shards = sorted(Path('datasets/tokenized/continual_medical_stage_b/train').glob('*.bin'))
assert len(stage_b_shards) == 107, f'Expected 107 Stage B shards, found {len(stage_b_shards)}'
assert Path('datasets/tokenized/evaluation_medical/dataset_manifest.json').is_file()
assert Path('datasets/tokenized/evaluation/dataset_manifest.json').is_file()
assert Path('artifacts/tokenizer/tokenizer.json').is_file()
print('Stage B shards:', len(stage_b_shards))
print('Parent checkpoint restored:', LOCAL_PARENT)

## Verify GPU, precision, tests, and parent initialization

In [ ]:
import torch
from medical_slm.training.precision import resolve_precision

assert torch.cuda.is_available(), 'Select a GPU runtime before continuing.'
gpu_name = torch.cuda.get_device_name(0)
precision = resolve_precision('auto', 'cuda')
print({'gpu': gpu_name, 'precision': precision.name, 'uses_grad_scaler': precision.uses_grad_scaler})

In [ ]:
subprocess.run([
    'python', '-m', 'pytest',
    'tests/test_stage_b_trainer.py',
    'tests/test_training_config.py',
    'tests/test_training_checkpoint.py',
    '-q',
], check=True)

In [ ]:
subprocess.run([
    'python', 'scripts/training/train_stage_b.py',
    '--config', 'configs/training_stage_b_colab.yaml',
    '--verify-initialization-only',
], check=True)
print('Parent/model/optimizer initialization gate: PASSED')

## Run the zero-update dual baseline

This evaluates the full medical validation split and the unchanged Stage A general validation split. Depending on the assigned GPU, it can take several minutes.

In [ ]:
subprocess.run([
    'python', 'scripts/training/train_stage_b.py',
    '--config', 'configs/training_stage_b_colab.yaml',
    '--baseline-only',
    '--baseline-output', str(LOCAL_REPORT),
], check=True)

## Verify and preserve the baseline report

In [ ]:
import json
import math

report = json.loads(LOCAL_REPORT.read_text())
medical = report['medical_validation']
general = report['general_validation']

assert report['status'] == 'passed'
assert report['optimizer_updates'] == 0
assert report['consumed_tokens'] == 0
assert report['model_parameters'] == 35_463_680
assert report['parent']['checkpoint_name'] == 'checkpoint_00007250'
assert medical['tokens'] == 997_632
assert general['tokens'] == 466_432
for result in (medical, general):
    assert math.isfinite(result['loss']) and math.isfinite(result['perplexity'])

DRIVE_REPORT.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(LOCAL_REPORT, DRIVE_REPORT)
assert DRIVE_REPORT.read_bytes() == LOCAL_REPORT.read_bytes()

print('STAGE B BASELINE GATE: PASSED')
print(f"Medical: loss={medical['loss']:.6f}, perplexity={medical['perplexity']:.3f}, tokens={medical['tokens']}")
print(f"General: loss={general['loss']:.6f}, perplexity={general['perplexity']:.3f}, tokens={general['tokens']}")
print(f"General-loss ceiling during Stage B: {report['general_loss_limit']:.6f}")
print('Saved:', DRIVE_REPORT)

## Pass criteria and next action

Proceed to the short Stage B development run only when the final cell prints `STAGE B BASELINE GATE: PASSED`. The two measured losses become immutable run baselines. During continual pretraining, a checkpoint is eligible for promotion only if medical validation improves and general validation loss remains no more than 5% above this baseline.